In [1]:
s1='Apple is looking at buying U.K. startup for $1 billion'

In [2]:
import spacy
nlp=spacy.load('en_core_web_sm')

In [3]:
doc=nlp(s1)

In [5]:
for token in doc:
    print(token.text,token.pos_,token.tag_,spacy.explain(token.tag_))

Apple PROPN NNP noun, proper singular
is AUX VBZ verb, 3rd person singular present
looking VERB VBG verb, gerund or present participle
at ADP IN conjunction, subordinating or preposition
buying VERB VBG verb, gerund or present participle
U.K. PROPN NNP noun, proper singular
startup VERB VBD verb, past tense
for ADP IN conjunction, subordinating or preposition
$ SYM $ symbol, currency
1 NUM CD cardinal number
billion NUM CD cardinal number


In [6]:
for key, value in doc.count_by(spacy.attrs.POS).items():
    print(key,doc.vocab[key].text,value)

96 PROPN 2
87 AUX 1
100 VERB 3
85 ADP 2
99 SYM 1
93 NUM 2


In [8]:
from spacy import displacy
displacy.render(doc,style='dep',options={'distance':80},jupyter=True)

In [9]:
s2='San Francise is looking at buying U.K. startup for $1 billion'
s3='facebook is hiring anew vice president in U.S.'

In [10]:
import spacy
nlp=spacy.load('en_core_web_sm')

In [11]:
doc1=nlp(s1)
for ent in doc1.ents:
    print(ent.text,ent.label_,str(spacy.explain(ent.label_)))

Apple ORG Companies, agencies, institutions, etc.
U.K. GPE Countries, cities, states
$1 billion MONEY Monetary values, including unit


In [12]:
doc3=nlp(s3)
for ent in doc3.ents:
    print(ent.text,ent.label_,str(spacy.explain(ent.label_)))

U.S. GPE Countries, cities, states


In [13]:
ORG = doc3.vocab.strings[u'ORG']
from spacy.tokens import Span
new_ent = Span(doc3, 0, 1, label=ORG)
doc3.ents = list(doc3.ents) + [new_ent]

In [14]:
doc3.ents

(facebook, U.S.)

In [15]:
for ent in doc3.ents:
    print(ent.text,ent.label_,str(spacy.explain(ent.label_)))

facebook ORG Companies, agencies, institutions, etc.
U.S. GPE Countries, cities, states


In [16]:
from spacy import displacy
displacy.render(doc1,style='ent',jupyter=True)

In [18]:
displacy.render(docs=doc1,style='ent',options={'ents':['ORG']},jupyter=True)

# Sentence Segmentation

In [1]:
s1='This is a sentence. This is second sentence. This is last sentence.'
s2='This is a sentence; This is second sentence; This is last sentence.'

In [2]:
import spacy
nlp=spacy.load('en_core_web_sm')

In [3]:
doc1=nlp(s1)

In [4]:
for sent in doc1.sents:
    print(sent.text)

This is a sentence.
This is second sentence.
This is last sentence.


In [5]:
s3='This is a sentence. This is second U.K. sentence. This is last sentence.'

In [6]:
doc3=nlp(s3)

In [7]:
for sent in doc3.sents:
    print(sent.text)

This is a sentence.
This is second U.K. sentence.
This is last sentence.


In [8]:
doc2=nlp(s2)
for sent in doc2.sents:
    print(sent.text)

This is a sentence; This is second sentence; This is last sentence.


In [9]:
def set_custom_boundaries(doc):
    for token in doc[:-1]:
        if token.text==';':
            doc[token.i+1].is_sent_start=True
    return doc

In [10]:
nlp.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [12]:
import spacy
from spacy.language import Language

@Language.component("set_custom_boundaries")
def set_custom_boundaries(doc):
    for token in doc[:-1]:
        if token.text == ';':
            doc[token.i+1].is_sent_start = True
    return doc

nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("set_custom_boundaries", before="parser")


<function __main__.set_custom_boundaries(doc)>

In [13]:
doc = nlp("Hello world; How are you?")
for sent in doc.sents:
    print(sent.text)


Hello world;
How are you?


In [14]:
nlp.pipe_names

['tok2vec',
 'tagger',
 'set_custom_boundaries',
 'parser',
 'attribute_ruler',
 'lemmatizer',
 'ner']

In [16]:
doc_2=nlp(s2)
for sent in doc_2.sents:
    print(sent.text)

This is a sentence;
This is second sentence;
This is last sentence.


# Spam Message Classification

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
import pandas as pd

data = {
    "id": [1, 2, 3],
    "text": ["Hello world!", "I am not happy today.", "Great job, well done!"],
    "label": ["positive", "negative", "positive"]
}

df = pd.DataFrame(data)

# UTF-8 encoding দিয়ে save করা
df.to_csv("sample_data.tsv", sep="\t", index=False, encoding="utf-8")


In [4]:
df = pd.read_csv("sample_data.tsv", sep="\t")

In [5]:
df.head()

,id,text,label
0,1,Hello world!,positive
1,2,I am not happy today.,negative
2,3,"Great job, well done!",positive


In [6]:
df.isna().sum()

,0
id,0
text,0
label,0


In [7]:
df.describe()

,id
count,3.0
mean,2.0
std,1.0
min,1.0
25%,1.5
50%,2.0
75%,2.5
max,3.0


In [9]:
df['label'].value_counts()/(len(df))

,count
label,
positive,0.666667
negative,0.333333


In [10]:
df[df['label']=='ham']

,id,text,label


In [11]:
positive=df[df['label']=='positive']
negative=df[df['label']=='negative']

In [12]:
positive

,id,text,label
0,1,Hello world!,positive
2,3,"Great job, well done!",positive


In [13]:
positive=positive.sample(negative.shape[0])

In [14]:
positive.shape,negative.shape

((1, 3), (1, 3))

In [16]:
data = pd.concat([positive, negative], ignore_index=True)

In [17]:
data.shape

(2, 3)

In [18]:
data['label'].value_counts()

,count
label,
positive,1
negative,1
